# CredX - Step 3: Model Training, Evaluation & Explainability
This notebook compares 3 classification models for default prediction:
1. Logistic Regression
2. Random Forest
3. XGBoost

Evaluates: Accuracy, Precision, Recall, F1 Score, ROC AUC.
Saves winning model (XGBoost) and SHAP TreeExplainer artifacts.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import shap

# Load engineered dataset
df = pd.read_csv("../data/processed/featured_dataset.csv")
print("Featured Dataset Shape:", df.shape)


## 1. Feature Preprocessing & Train/Test Split


In [ ]:
y = df['defaulted'].astype(int)
drop_cols = ['borrower_id', 'defaulted', 'default_probability']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[numerical_cols])
X_test_num = scaler.transform(X_test[numerical_cols])

X_train_proc = np.hstack([X_train_num, X_train_cat])
X_test_proc = np.hstack([X_test_num, X_test_cat])
print("Processed Training Matrix Shape:", X_train_proc.shape)


## 2. Model Training & Comparison


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=250, max_depth=5, learning_rate=0.04, eval_metric='logloss', random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_proc, y_train)
    preds = model.predict(X_test_proc)
    probs = model.predict_proba(X_test_proc)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'F1 Score': f1_score(y_test, preds, zero_division=0),
        'ROC AUC': roc_auc_score(y_test, probs)
    })

pd.DataFrame(results).set_index('Model').round(4)


## 3. Alternative Credit Scoring Formula
- Default Probability $p = \text{predict_proba}(X)[1]$
- Alternate Credit Score $= 300 + ((1 - p) \times 600) \quad [300 - 900]$
- Risk Bands: Low ($<0.30$), Medium ($0.30 - 0.60$), High ($\ge 0.60$)
- Approval Probability $= (1 - p) \times 100\%$


In [ ]:
best_model = models['XGBoost']
sample_prob = best_model.predict_proba(X_test_proc[:5])[:, 1]
for i, prob in enumerate(sample_prob):
    score = int(round(300 + (1 - prob) * 600))
    risk = "Low Risk" if prob < 0.30 else ("Medium Risk" if prob < 0.60 else "High Risk")
    approval = round((1 - prob) * 100, 1)
    print(f"Applicant {i+1}: Default Prob={prob:.3f} | Score={score} | Risk={risk} | Approval={approval}%")


## 4. SHAP Explainability Demo


In [ ]:
explainer = shap.TreeExplainer(best_model)
sample_shap = explainer.shap_values(X_test_proc[:1])
print("Sample SHAP Values calculated successfully. Shape:", sample_shap.shape)
